In [0]:

%run ./01.config.py

In [0]:
# MAGIC %run ./00_config

# COMMAND ----------
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze = spark.table(BRONZE)
for c in ["operation_type", "quantity_unit", "product_name", "destination_city"]:
    display(bronze.groupBy(c).count().orderBy(F.desc("count")))   # dirty values yahan dikhenge

# COMMAND ----------
df = bronze

# 1. Numeric casts
num_cols = ["ordered_quantity","demand_quantity","available_inventory","unit_price_usd",
            "product_cost_usd","transportation_cost_usd","total_cost_usd",
            "supplier_reliability_score","quality_score",
            "expected_lead_time_days","actual_lead_time_days","delay_days"]
for c in num_cols:
    df = df.withColumn(c, F.regexp_replace(F.col(c), ",", "").cast("double"))
df = df.withColumn("ingestion_timestamp", F.to_timestamp("ingestion_timestamp"))

# 2. Boolean flags -> 0/1
for c in ["is_delayed", "is_stockout"]:
    df = df.withColumn(c, F.when(F.lower(F.trim(F.col(c))).isin("true","1","yes","y"), 1).otherwise(0))

# 3. Text standardise
for c in ["product_name","product_category","destination_city","supplier_name",
          "transportation_mode","quantity_unit"]:
    df = df.withColumn(c, F.initcap(F.trim(F.col(c))))
df = df.withColumn("operation_type", F.upper(F.trim("operation_type")))
df = df.replace({"Bombay":"Mumbai","Poona":"Pune","New Delhi":"Delhi","Calcutta":"Kolkata"},
                subset=["destination_city"])
# TODO: distinct values dekh kar aur typos add karo. Agar quantity_unit mixed hai to yahan barrels mein convert karo.

# 4. Latest record per transaction_id, deletes hatao
w = Window.partitionBy("transaction_id").orderBy(F.col("ingestion_timestamp").desc())
df = (df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")
        .filter(F.col("operation_type") != "DELETE"))

# 5. Calendar columns date se derive karo
df = (df.withColumn("transaction_year", F.year("txn_date"))
        .withColumn("transaction_quarter", F.quarter("txn_date"))
        .withColumn("transaction_month", F.month("txn_date")))

# 6. DQ rules (condition = BAD)
rules = {
    "null_txn_id":     F.col("transaction_id").isNull(),
    "null_date":       F.col("txn_date").isNull(),
    "future_date":     F.col("txn_date") > F.current_date(),
    "null_product":    F.col("product_name").isNull() | (F.col("product_name") == ""),
    "null_city":       F.col("destination_city").isNull() | (F.col("destination_city") == ""),
    "null_demand":     F.col("demand_quantity").isNull(),
    "negative_demand": F.col("demand_quantity") < 0,
    "negative_price":  F.col("unit_price_usd") < 0,
}
df = df.withColumn("_dq_failures", F.concat_ws(",", *[F.when(c, F.lit(n)) for n, c in rules.items()]))
good = df.filter(F.col("_dq_failures") == "").drop("_dq_failures", "_rescued_data")
bad  = df.filter(F.col("_dq_failures") != "")

good.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(SILVER)
bad.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(SILVER_REJECTED)

# COMMAND ----------
print("bronze:", bronze.count(), "| silver:", spark.table(SILVER).count(), "| rejected:", spark.table(SILVER_REJECTED).count())
display(spark.table(SILVER_REJECTED).groupBy("_dq_failures").count().orderBy(F.desc("count")))